# Task 2 — Symbolic, conditioned generation (POP909 chord-conditioned melody)

We evaluate **chord-conditioned monophonic melody generation**: given a chord progression (`root, quality, beat, bar` per 16th note) the model emits melody tokens (`0-127` pitch onset, `128` REST, `129` HOLD).

We compare our learned conditioned models against **trivial baselines** on the held-out **test** split, and save all charts / metrics / a written report to `eval/task2/`. Heavy logic lives in `eval/task2/eval_task2.py` so the notebook stays clean and never re-runs training.

In [ ]:
# generate for baseline
#
# Trivial (non-learned) baselines, both proper conditional models p(token|chord):
#   * Unigram (no chord): i.i.d. from empirical token frequency (ignores chords)
#   * Chord-tone rule    : memoryless; rhythm from marginal rates, onset pitch
#                          drawn from empirical pitches restricted to the chord's
#                          tones (uses the chord in the most naive way)
import os, sys
import numpy as np

EVAL_SRC = os.path.join(os.getcwd(), "eval", "task2")
if EVAL_SRC not in sys.path:
    sys.path.insert(0, EVAL_SRC)
import eval_task2 as E   # self-contained: baselines, models, metrics, plots

rng = np.random.default_rng(E.SEED)
train_chord, train_melody = E.load_split("train")
test_chord,  test_melody  = E.load_split("test")

baselines = {
    "Unigram (no chord)": E.UnigramBaseline(train_melody),
    "Chord-tone rule":    E.ChordToneRuleBaseline(train_melody),
}

# Generate a baseline melody for one TEST chord progression and save it as MIDI
idx = 0
rule_tokens = E.baseline_generate(baselines["Chord-tone rule"], test_chord[idx], rng)
E.melody_to_midi(rule_tokens, os.path.join(E.EVAL_DIR, "baseline_chord_tone_rule.mid"))
print("Chord-tone-rule baseline (first 16 tokens):", rule_tokens[:16].tolist())
print("Saved ->", os.path.join(E.EVAL_DIR, "baseline_chord_tone_rule.mid"))

In [ ]:
# generate for our model
#
# Our method has two chord-conditioned models (chord_gru and chord_transformer,
# both count as ours). Here we decode with the Transformer on the SAME test
# chord progression so baseline and our model are directly comparable;
# the evaluation cell below scores BOTH of our models against the baselines.
import os, torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, ckpt = E.load_checkpoint(os.path.join(E.CKPT_DIR, "chord_transformer_best.pt"), device)
print(f"Loaded {ckpt['model_name']} (epoch {ckpt.get('epoch')}, val_loss {ckpt.get('val_loss'):.4f})")

our_tokens = E.nn_generate_tokens(model, test_chord[idx], device, decoding="top_k", top_k=8)
E.melody_to_midi(our_tokens, os.path.join(E.EVAL_DIR, "our_model_chord_transformer.mid"))
# Render the REAL melody for the same chords too (A/B listening reference)
E.melody_to_midi(test_melody[idx], os.path.join(E.EVAL_DIR, "real_reference.mid"))
print("Our model melody (first 16 tokens):", [int(t) for t in our_tokens[:16]])
print("Saved -> our_model_chord_transformer.mid  (+ real_reference.mid)")

In [ ]:
# evaluate
#
# One protocol for every method on the held-out TEST split:
#   (1) conditional perplexity   (2) chord-tone ratio (harmony, vs real)
#   (3) pitch-class / interval / bar-onset JS divergence to real
#   (4) rest ratio + onset density (rhythm realism)
# Saves charts, metrics.json and evaluation_writeup.txt to eval/task2/.
from IPython.display import Image, display

metrics = E.run_full_evaluation(n_eval=96, decoding="top_k", top_k=8)

print("\nTest perplexity (lower = better):")
for m, v in metrics["perplexity"].items():
    print(f"   {m:26s} {v:7.3f}")
print(f"\nChord-tone ratio (real = {metrics['real']['chord_tone_ratio']:.3f}):")
for m, v in metrics["chord_tone_ratio"].items():
    print(f"   {m:26s} {v:.3f}")

for png in ["perplexity_comparison.png", "chord_tone_ratio.png",
            "js_divergence_summary.png", "rhythm_realism.png",
            "pitch_class_distribution.png"]:
    display(Image(filename=os.path.join(E.EVAL_DIR, png)))

print(open(os.path.join(E.EVAL_DIR, "evaluation_writeup.txt")).read())